In [ ]:
# @title Google Colab setup (run this cell first)  {display-mode: "form"}
import os, sys

# Clone the repo if not already present
if not os.path.isdir("dl4bi"):
    os.system(
        "git clone --branch claude/deeprv-nonnegative-tutorial-QQCvh "
        "https://github.com/flaxter/dl4bi.git"
    )

# Add the package root to sys.path.
# We do NOT use `pip install -e .` because pyproject.toml lists `sps` as an
# SSH-only dependency; we install it below via HTTPS instead.
repo_root = os.path.abspath("dl4bi")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)

# Colab already has: jax, numpy, matplotlib, scipy, tqdm, optax, orbax, wandb.
# We only need to upgrade flax (0.11 → ≥0.12) and install a few missing packages.
os.system("pip install -q 'flax>=0.12' 'hydra-core>=1.3' numpyro jraph einops orbax-checkpoint scoringrules git+https://github.com/MLGlobalHealth/sps.git")

print("Setup complete.")
print(f"Working directory: {os.getcwd()}")
print(f"sys.path[0]:       {sys.path[0]}")

# Non-negative Bivariate Normal Sampling with DeepRV

## Overview

This tutorial introduces **DeepRV** — a decoder-only neural surrogate that emulates samples
from a stochastic process — in the simplest non-trivial setting: a **bivariate Normal** distribution
whose samples are mapped to the non-negative orthant via the **softplus** transformation.

### The generative model

$$
\rho \sim \text{Uniform}(-0.99,\, 0.99), \qquad
\mathbf{z} \sim \mathbf{N}(\mathbf{0}, \mathbf{I}_2)
$$
$$
\mathbf{f} = L(\rho)\,\mathbf{z}, \qquad
\boldsymbol{\mu} = \text{softplus}(\mathbf{f}) \geq 0
$$
$$
\mathbf{y} \sim \mathbf{N}(\boldsymbol{\mu},\, \sigma_{\text{obs}}^2 \mathbf{I})
$$

where $L(\rho) = \text{Chol}\bigl(K(\rho)\bigr)$ and
$K(\rho) = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix}$.

### What DeepRV does

DeepRV learns the mapping
$$
\hat{\mathbf{f}} = \text{DeepRV}(\mathbf{z},\, \rho) \approx L(\rho)\,\mathbf{z}
$$
so that it can be dropped in as a **differentiable** surrogate inside a NumPyro model,
enabling gradient-based MCMC (NUTS) without re-computing the Cholesky factor at each step.

For a 2×2 matrix the Cholesky is trivially cheap — this tutorial is **pedagogical**:
the same pattern scales to thousands of locations where Cholesky costs $O(N^3)$.

### Tutorial outline
1. Imports & constants  
2. Bivariate covariance and training data  
3. Train the DeepRV surrogate  
4. Generate synthetic observations  
5. Bayesian inference via NUTS (DeepRV vs exact Cholesky)  
6. Visualise results

## 1. Imports & constants

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import optax
import wandb
from jax import jit, random
from jax.nn import softplus
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median

from dl4bi.core.model_output import VAEOutput
from dl4bi.core.train import cosine_annealing_lr, train
from dl4bi.vae import gMLPDeepRV
from dl4bi.vae.train_utils import deep_rv_train_step, generate_surrogate_decoder

wandb.init(mode="disabled")

In [ ]:
# ── problem constants ──────────────────────────────────────────────────────────
N_LOCS     = 2      # bivariate: exactly 2 components
TRUE_RHO   = 0.8    # correlation we want to recover
NOISE_STD  = 0.2    # observation noise σ_obs
N_OBS      = 50     # number of noisy observations of μ
SEED       = 42

# ── "locations" for the two variables ─────────────────────────────────────────
# gMLPDeepRV is a spatial model; we represent the two components as
# points at positions 0 and 1 on a 1-D axis.
s = jnp.array([[0.], [1.]])   # shape (2, 1)

# keyword-args passed to the surrogate decoder at inference time
surrogate_kwargs = {"s": s}

## 2. Bivariate covariance and training data

The bivariate correlation matrix is

$$K(\rho) = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix}$$

Training data consists of `(z, f)` pairs drawn for random $\rho$ values:
$$\rho \sim \text{Uniform}(-0.99, 0.99), \quad
\mathbf{z} \sim \mathbf{N}(0, I_2), \quad
\mathbf{f} = L(\rho)\,\mathbf{z}$$

DeepRV is trained to recover $\mathbf{f}$ from $(\mathbf{z}, \rho)$.

In [ ]:
@jit
def biv_cholesky(rho, jitter=1e-4):
    """Cholesky factor of the 2×2 correlation matrix parameterised by rho."""
    K = jnp.array([[1., rho], [rho, 1.]]) + jitter * jnp.eye(2)
    return jnp.linalg.cholesky(K)


priors = {"rho": dist.Uniform(-0.99, 0.99)}


def gen_train_dataloader(s, priors, batch_size=64):
    """
    Infinite generator of training batches.

    Each batch samples a fresh ρ and returns:
      z           – standard-normal noise,  shape (batch, 2)
      f           – GP realization L(ρ)·z,  shape (batch, 2)   ← training target
      conditionals – [ρ],                   shape (1,)
      s           – location array,          shape (2, 1)
    """
    def dataloader(rng_data):
        while True:
            rng_data, rng_rho, rng_z = random.split(rng_data, 3)
            rho = priors["rho"].sample(rng_rho)
            z   = dist.Normal().sample(rng_z, sample_shape=(batch_size, N_LOCS))
            L   = biv_cholesky(rho)
            f   = jnp.einsum("ij,bj->bi", L, z)   # (batch, 2)
            yield {
                "s": s,
                "z": z,
                "conditionals": jnp.array([rho]),
                "f": f,
            }
    return dataloader


@jit
def valid_step(rng, state, batch):
    output: VAEOutput = state.apply_fn(
        {"params": state.params, **state.kwargs}, **batch, rngs={"extra": rng}
    )
    return {"norm MSE": output.metrics(batch["f"], 1.0)["MSE"]}

## 3. Train the DeepRV surrogate

`gMLPDeepRV` is a gated-MLP decoder: it takes the latent noise `z` and the
conditional hyperparameter `ρ`, then predicts the GP realization `f`.
Training minimises MSE between the predicted and true Cholesky-transformed samples.

In [ ]:
from dl4bi.core.train import Callback

rng = random.key(SEED)
rng_train, rng_obs, rng_infer_drv, rng_infer_exact, rng_vis = random.split(rng, 5)

# ── fixed test batch used by the callback ─────────────────────────────────────
# Same z and ρ throughout training so RMSE is a fair comparison across steps.
rng_cb     = random.fold_in(rng_vis, 101)
n_cb       = 300
z_cb       = dist.Normal().sample(rng_cb, (n_cb, N_LOCS))
rho_cb     = 0.8
f_cb_exact = jnp.einsum("ij,bj->bi", biv_cholesky(rho_cb), z_cb)  # ground truth

# ── storage ───────────────────────────────────────────────────────────────────
curve_steps = []
curve_rmse  = []
snapshots   = {}   # step -> (params, kwargs)  for scatter plots later

SNAPSHOT_STEPS = {1_000, 5_000, 15_000, 30_000}

nn_model = gMLPDeepRV(num_blks=2)

def record_progress(step, rng_step, state, batch, extra):
    """Callback: evaluate RMSE on a fixed held-out batch every N steps."""
    f_hat = nn_model.apply(
        {"params": state.params, **state.kwargs},
        z_cb, jnp.array([rho_cb]), s=s,
        rngs={"extra": rng_step},
    ).f_hat.squeeze(-1)                                           # (n_cb, 2)
    rmse = float(jnp.sqrt(jnp.mean((f_hat - f_cb_exact) ** 2)))
    curve_steps.append(step)
    curve_rmse.append(rmse)
    if step in SNAPSHOT_STEPS:
        snapshots[step] = (state.params, state.kwargs)

# ── training ──────────────────────────────────────────────────────────────────
optimizer = optax.chain(
    optax.clip_by_global_norm(3.0),
    optax.adamw(cosine_annealing_lr(30_000, 1e-3), weight_decay=1e-2),
)
loader = gen_train_dataloader(s, priors)

state = train(
    rng_train,
    nn_model,
    optimizer,
    deep_rv_train_step,
    30_000,
    loader,
    valid_step,
    5_000,
    500,
    loader,
    return_state="best",
    valid_monitor_metric="norm MSE",
    callbacks=[Callback(fn=record_progress, interval=300)],
)

surrogate_decoder = generate_surrogate_decoder(state, nn_model)
print("DeepRV surrogate training complete.")

### Training curve: how approximation quality evolves

The callback recorded RMSE on a fixed held-out batch (300 samples, ρ = 0.8) every 300 steps. Below we plot the curve and four scatter snapshots — one per `SNAPSHOT_STEPS` — showing `f̂₂` vs `f₂` (component 2, the non-trivial learnable mapping). At step 1 000 the scatter is diffuse; by step 30 000 it collapses onto the diagonal.

In [ ]:
snap_steps_sorted = sorted(snapshots.keys())

fig = plt.figure(figsize=(18, 9))
gs  = fig.add_gridspec(2, len(snap_steps_sorted), hspace=0.45, wspace=0.3)

# ── top row: full-width training curve ────────────────────────────────────────
ax_curve = fig.add_subplot(gs[0, :])
ax_curve.plot(curve_steps, curve_rmse, color="steelblue", lw=1.5)
for step in snap_steps_sorted:
    ax_curve.axvline(step, color="crimson", ls="--", lw=1, alpha=0.7)
    ax_curve.text(step, max(curve_rmse) * 0.97, f" {step:,}", color="crimson",
                  fontsize=8, va="top")
ax_curve.set_xlabel("Training step")
ax_curve.set_ylabel("RMSE  (held-out batch, ρ = 0.8)")
ax_curve.set_title("Training curve: surrogate approximation error over time")
ax_curve.set_yscale("log")
ax_curve.grid(True, which="both", alpha=0.3)

# ── bottom row: scatter snapshots ─────────────────────────────────────────────
# Show component 2 (f₂ = 0.8 z₁ + 0.6 z₂), the non-trivial learnable part.
for col, step in enumerate(snap_steps_sorted):
    params, kwargs = snapshots[step]
    f_hat_snap = nn_model.apply(
        {"params": params, **kwargs},
        z_cb, jnp.array([rho_cb]), s=s,
        rngs={"extra": random.key(0)},
    ).f_hat.squeeze(-1)                                           # (n_cb, 2)

    f_ex2  = f_cb_exact[:, 1]   # exact  f₂
    f_dr2  = f_hat_snap[:, 1]   # DeepRV f̂₂
    rmse2  = float(jnp.sqrt(jnp.mean((f_dr2 - f_ex2) ** 2)))
    lo, hi = float(min(f_ex2.min(), f_dr2.min())), float(max(f_ex2.max(), f_dr2.max()))

    ax = fig.add_subplot(gs[1, col])
    ax.scatter(f_ex2, f_dr2, alpha=0.3, s=10, c="steelblue", rasterized=True)
    ax.plot([lo, hi], [lo, hi], color="crimson", lw=1.5, ls="--")
    ax.set_title(f"Step {step:,}\nRMSE = {rmse2:.4f}", fontsize=10)
    ax.set_xlabel(r"$f_2$ (exact)", fontsize=9)
    if col == 0:
        ax.set_ylabel(r"$\hat{f}_2$ (DeepRV)", fontsize=9)

plt.suptitle(
    r"Snapshot scatter: $\hat{f}_2$ vs $f_2 = \rho\,z_1 + \sqrt{1-\rho^2}\,z_2$  at  ρ = 0.8",
    fontsize=11, y=1.01,
)
plt.savefig("deeprv_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

### Approximation quality: $\hat{\mathbf{f}}$ vs exact $L(\rho)\,\mathbf{z}$

For a 2×2 matrix the Cholesky has a closed form:

$$L(\rho) = \begin{pmatrix}1 & 0\\ \rho & \sqrt{1-\rho^2}\end{pmatrix}
\;\implies\;
f_1 = z_1, \qquad f_2 = \rho\,z_1 + \sqrt{1-\rho^2}\,z_2$$

We test the surrogate **directly**: feed the **same** $\mathbf{z}$ vectors to both the exact transform and DeepRV, then scatter $\hat{f}_i$ against $f_i$. A perfect surrogate puts every point on the diagonal $\hat{f} = f$.

The grid below sweeps $\rho \in \{-0.8, -0.4, 0, 0.4, 0.8\}$ and shows both components. Component 1 ($f_1 = z_1$) carries no $\rho$-dependence; component 2 ($f_2 = \rho z_1 + \sqrt{1-\rho^2}\,z_2$) is the non-trivial learnable part.

In [ ]:
rng_check = random.fold_in(rng_vis, 99)   # fresh, reproducible key
n_check   = 500
z_check   = dist.Normal().sample(rng_check, (n_check, N_LOCS))  # same z for every ρ

rho_grid  = [-0.8, -0.4, 0.0, 0.4, 0.8]
fig, axes = plt.subplots(2, len(rho_grid), figsize=(16, 6), sharey="row")

component_formulas = [
    r"$f_1 = z_1$",
    r"$f_2 = \rho z_1 + \sqrt{1{-}\rho^2}\,z_2$",
]

for col, rho in enumerate(rho_grid):
    L_rho   = biv_cholesky(rho)
    f_exact = jnp.einsum("ij,bj->bi", L_rho, z_check)                                     # (500, 2)
    f_drv   = surrogate_decoder(z_check, jnp.array([rho]), **surrogate_kwargs).squeeze(-1)  # (500, 2)

    for row in range(2):
        ax   = axes[row, col]
        lo   = float(min(f_exact[:, row].min(), f_drv[:, row].min()))
        hi   = float(max(f_exact[:, row].max(), f_drv[:, row].max()))
        rmse = float(jnp.sqrt(jnp.mean((f_drv[:, row] - f_exact[:, row]) ** 2)))

        ax.scatter(f_exact[:, row], f_drv[:, row],
                   alpha=0.25, s=8, c="steelblue", rasterized=True)
        ax.plot([lo, hi], [lo, hi], color="crimson", lw=1.5, ls="--")
        ax.text(0.05, 0.91, f"RMSE = {rmse:.4f}",
                transform=ax.transAxes, fontsize=8)
        if row == 0:
            ax.set_title(f"ρ = {rho:.1f}", fontsize=11)
        ax.set_xlabel(f"$f_{row + 1}$  (exact)", fontsize=9)
        if col == 0:
            ax.set_ylabel(
                f"$\\hat{{f}}_{row + 1}$  (DeepRV)\n{component_formulas[row]}",
                fontsize=9,
            )

plt.suptitle(
    r"DeepRV approximation quality: $\hat{\mathbf{f}}$ vs $L(\rho)\,\mathbf{z}$"
    "   —   points should lie on the red diagonal",
    fontsize=11,
)
plt.tight_layout()
plt.show()

# ── pre-compute vis arrays used in the final results plot ─────────────────────
n_vis       = 1_000
z_vis       = dist.Normal().sample(rng_vis, (n_vis, N_LOCS))
L_true      = biv_cholesky(TRUE_RHO)
f_exact_vis = jnp.einsum("ij,bj->bi", L_true, z_vis)
y_exact_vis = softplus(f_exact_vis)
f_drv_vis   = surrogate_decoder(z_vis, jnp.array([TRUE_RHO]), **surrogate_kwargs).squeeze(-1)
y_drv_vis   = softplus(f_drv_vis)

## 4. Generate synthetic observations

We draw a single latent pair $\mathbf{f}_0 = L(\rho^*)\,\mathbf{z}_0$ and observe
it with additive Gaussian noise $N_{\text{obs}} = 50$ times:

$$\mathbf{y}^{(k)} \sim \mathbf{N}\!\left(\text{softplus}(\mathbf{f}_0),\, \sigma_{\text{obs}}^2 I\right),
\quad k = 1,\ldots,N_{\text{obs}}$$

The inference goal is to recover $\rho^* = 0.8$ from these non-negative observations.

In [ ]:
rng_z0, rng_noise = random.split(rng_obs)

z0       = dist.Normal().sample(rng_z0, (1, N_LOCS))                        # (1, 2)
f0       = jnp.einsum("ij,bj->bi", biv_cholesky(TRUE_RHO), z0)[0]          # (2,)
mu_true  = softplus(f0)                                                       # (2,) ≥ 0
y_obs    = mu_true + dist.Normal(0., NOISE_STD).sample(rng_noise, (N_OBS, N_LOCS))

print(f"True correlation:    ρ* = {TRUE_RHO}")
print(f"True latent pair:    f₀ = {f0.tolist()}")
print(f"True non-neg mean:   μ  = softplus(f₀) = {mu_true.tolist()}")
print(f"Observed data shape: y_obs ~ {y_obs.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(y_obs[:, 0], y_obs[:, 1], alpha=0.6, s=25, c="gray", label="observations")
ax.scatter(*mu_true, color="crimson", s=120, marker="*", zorder=10, label=f"true μ")
ax.set_xlabel(r"$y_1$  (non-negative)")
ax.set_ylabel(r"$y_2$  (non-negative)")
ax.set_title(f"Synthetic observations  (N={N_OBS}, σ_obs={NOISE_STD})")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Bayesian inference via NUTS

We define a single NumPyro model that can use **either** the DeepRV surrogate
**or** the exact Cholesky, controlled by the `surrogate_decoder` argument.
This makes it easy to compare both approaches on identical data.

In [ ]:
def bivariate_nonneg_model(surrogate_decoder=None, y=None):
    """
    NumPyro model for non-negative bivariate Normal observations.

    Latent variables
    ----------------
    rho : scalar        – bivariate correlation
    z   : (1, 2) array  – standard-normal latent noise

    Deterministic quantities
    ------------------------
    f   : (2,)  – GP realization  L(rho) · z[0]  (or DeepRV surrogate)
    mu  : (2,)  – non-negative mean  softplus(f)

    Likelihood
    ----------
    y   : (N_OBS, 2)  – Normal(mu, NOISE_STD²),  observed
    """
    rho = numpyro.sample("rho", dist.Uniform(-0.99, 0.99))
    z   = numpyro.sample("z",   dist.Normal(), sample_shape=(1, N_LOCS))

    if surrogate_decoder is None:
        # ── exact Cholesky ───────────────────────────────────────────────────
        L = biv_cholesky(rho)
        f = numpyro.deterministic("f", jnp.einsum("ij,bj->bi", L, z)[0])
    else:
        # ── DeepRV surrogate ─────────────────────────────────────────────────
        # z: (1, 2) → decode → (1, 2, 1) → squeeze → (2,)
        f = numpyro.deterministic(
            "f",
            surrogate_decoder(z, jnp.array([rho]), **surrogate_kwargs).squeeze(),
        )

    mu = numpyro.deterministic("mu", softplus(f))          # (2,) ≥ 0
    # mu broadcasts over the N_OBS dimension of y
    numpyro.sample("obs", dist.Normal(mu, NOISE_STD), obs=y)

In [ ]:
def run_nuts(rng_key, surrogate_decoder, y, label=""):
    nuts = NUTS(bivariate_nonneg_model, init_strategy=init_to_median(num_samples=10))
    mcmc = MCMC(nuts, num_chains=2, num_samples=500, num_warmup=500)
    mcmc.run(rng_key, surrogate_decoder=surrogate_decoder, y=y)
    print(f"\n{'─'*50}  {label}")
    mcmc.print_summary()
    return mcmc.get_samples()

In [ ]:
samples_drv   = run_nuts(rng_infer_drv,   surrogate_decoder, y_obs, "DeepRV")

In [ ]:
samples_exact = run_nuts(rng_infer_exact, None,              y_obs, "Exact Cholesky")

## 6. Results

We visualise three things:
1. **Posterior of ρ** — does DeepRV match the exact posterior?
2. **Observation scatter** — the non-negative data we conditioned on
3. **Sample quality** — do DeepRV samples look like exact Cholesky samples?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── (a) Posterior of ρ ────────────────────────────────────────────────────────
ax = axes[0]
ax.hist(
    samples_drv["rho"],   bins=40, density=True,
    alpha=0.6, color="steelblue",  label="DeepRV"
)
ax.hist(
    samples_exact["rho"], bins=40, density=True,
    alpha=0.6, color="darkorange", label="Exact Cholesky"
)
ax.axvline(
    TRUE_RHO, color="crimson", lw=2, ls="--",
    label=f"True ρ = {TRUE_RHO}"
)
ax.set_xlabel("Correlation ρ")
ax.set_ylabel("Posterior density")
ax.set_title("Posterior: Bivariate Correlation")
ax.legend()

# ── (b) Observed non-negative pairs ──────────────────────────────────────────
ax = axes[1]
ax.scatter(y_obs[:, 0], y_obs[:, 1], alpha=0.55, s=25, c="gray", label="observations")
ax.scatter(*mu_true, color="crimson", s=120, marker="*", zorder=10, label="true μ")
ax.set_xlabel(r"$y_1$  (non-negative)")
ax.set_ylabel(r"$y_2$  (non-negative)")
ax.set_title(f"Synthetic Data  (N={N_OBS}, σ_obs={NOISE_STD})")
ax.legend()

# ── (c) Sample quality at TRUE_RHO ───────────────────────────────────────────
ax = axes[2]
ax.scatter(
    y_exact_vis[:, 0], y_exact_vis[:, 1],
    alpha=0.25, s=6, c="darkorange", label="Exact"
)
ax.scatter(
    y_drv_vis[:, 0],   y_drv_vis[:, 1],
    alpha=0.25, s=6, c="steelblue",  label="DeepRV"
)
ax.set_xlabel(r"$y_1 = \mathrm{softplus}(f_1)$")
ax.set_ylabel(r"$y_2 = \mathrm{softplus}(f_2)$")
ax.set_title(f"Sample Quality at ρ = {TRUE_RHO}")
ax.legend(markerscale=3)

plt.tight_layout()
plt.savefig("deeprv_nonneg_bivariate_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to deeprv_nonneg_bivariate_results.png")

## Summary

| | DeepRV | Exact Cholesky |
|---|---|---|
| **Posterior agreement** | ✓ (matches exact) | reference |
| **Complexity per MCMC step** | O(N) forward pass | O(N³) Cholesky |
| **Differentiable** | ✓ (JAX autodiff) | ✓ |
| **Requires training** | ✓ (one-off offline) | ✗ |

The key takeaway:
- **DeepRV replaces the Cholesky** inside the NumPyro model, turning each HMC
  gradient evaluation from $O(N^3)$ into a cheap neural forward pass.
- **Softplus** provides a smooth, everywhere-differentiable map from the
  unconstrained GP sample $\mathbf{f}$ to the non-negative orthant.
- The surrogate is trained **once** and then reused for any number of inference
  runs with different observed data.

### Scaling up
Replace `s = jnp.array([[0.], [1.]])` with a full spatial grid (e.g. `build_grid(...)` from `sps.utils`),
update the covariance kernel to a Matérn or RBF function, and the rest of the code is
**identical** — see `benchmarks/vae/deep_rv_example.py` for a 256-location example.